# SpatialLens Assist - Week 1 / 2 sanity check

Run the pipeline first, then run all cells in this notebook to inspect
the detections, tracks, motion features, plots, and the text summary for a
single `video_id` (defaults to one of the mock scenarios).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

VIDEO_ID = 'bike_approaching_left'

DETECTIONS_DIR = REPO_ROOT / 'outputs' / 'detections'
TRACKS_DIR = REPO_ROOT / 'outputs' / 'tracks'
ANNOTATED_FRAMES_DIR = REPO_ROOT / 'outputs' / 'annotated_frames'
PLOTS_DIR = REPO_ROOT / 'outputs' / 'plots'
SUMMARIES_DIR = REPO_ROOT / 'outputs' / 'summaries'

## Load detections + track features

In [ ]:
import pandas as pd

det_csv = DETECTIONS_DIR / f'{VIDEO_ID}_detections.csv'
feat_csv = TRACKS_DIR / f'{VIDEO_ID}_track_features.csv'
tracks_csv = TRACKS_DIR / f'{VIDEO_ID}_tracks.csv'

detections = pd.read_csv(det_csv) if det_csv.exists() else pd.DataFrame()
track_features = pd.read_csv(feat_csv) if feat_csv.exists() else pd.DataFrame()
tracks = pd.read_csv(tracks_csv) if tracks_csv.exists() else pd.DataFrame()

print('detections shape:', detections.shape)
print('tracks shape:', tracks.shape)
print('track features shape:', track_features.shape)

## Detection counts by class

In [ ]:
if detections.empty:
    print('No detections found.')
else:
    display(detections['class_name'].value_counts().to_frame('count'))

## Show 3 annotated frames (tracking overlay)

In [ ]:
import matplotlib.pyplot as plt
import cv2

track_frame_dir = ANNOTATED_FRAMES_DIR / f'{VIDEO_ID}_tracks'
frame_files = sorted(track_frame_dir.glob('frame_*.jpg'))
if not frame_files:
    print(f'No annotated tracking frames at {track_frame_dir}')
else:
    picks = [frame_files[0], frame_files[len(frame_files)//2], frame_files[-1]]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, fp in zip(axes, picks):
        img = cv2.imread(str(fp))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(fp.name)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Plot track trajectories + bbox area over time

In [ ]:
if tracks.empty:
    print('No tracks to plot.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for tid, g in tracks.groupby('track_id'):
        g = g.sort_values('frame_idx')
        axes[0].plot(g['cx'], g['cy'], marker='o', label=str(tid))
        axes[1].plot(g['frame_idx'], g['area'], marker='o', label=str(tid))
    axes[0].invert_yaxis()
    axes[0].set_title('Track trajectories (image coords)')
    axes[0].set_xlabel('cx (px)')
    axes[0].set_ylabel('cy (px)')
    axes[0].legend(fontsize=8)
    axes[1].set_title('Bbox area over time')
    axes[1].set_xlabel('frame_idx')
    axes[1].set_ylabel('bbox area (px^2)')
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Week 1-2 text summary

In [ ]:
summary_path = SUMMARIES_DIR / f'{VIDEO_ID}_week1_week2_summary.txt'
if summary_path.exists():
    print(summary_path.read_text())
else:
    print(f'No summary found at {summary_path}')